In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
train = pd.read_csv('train.csv')
train = train
train.head(6)

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male
5,5,at-risk,5.11,82.8,24.43,2290.0,2134.0,29.2,2.02,veg,low,poor,sedentary,occasional,male


In [3]:
# Feature Engineering Discoveries
train['diet_type'].unique()
train['gender'].unique()
train.isna().sum()

id                             0
health_condition               0
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64

In [88]:
# Feature Engineering 
from sklearn.preprocessing import FunctionTransformer
def feature_engineering(df):
    df = df.copy()
    df['sleep_deprived'] = np.where(
        df['sleep_duration'].isna(),
        np.nan,
        (df['sleep_duration'] < 7).astype(int)
    )
    df['bmi_category'] = pd.cut(
        df['bmi'],
        bins = [0, 18.5, 24.9, 29.9, 99999999999],
        labels = ['underweight', 'healthy_weight', 'over_weight', 'obese']
    )

    df['calories_per_minute'] = (df['calorie_expenditure'].replace(0, np.nan) /
        df['exercise_duration'].replace(0, np.nan))
    
    df['water_per_bmi'] = (df['water_intake'].replace(0, np.nan) / 
                           df['bmi'].replace(0, np.nan))
    df['steps_per_minute'] = (df['step_count'].replace(0, np.nan) /
            df['exercise_duration'].replace(0, np.nan)
    )
    df['heart_rate_category'] = pd.cut(
        df['heart_rate'],
        bins = [0, 60, 80, 100,  df['heart_rate'].max()],
        labels = ['low', 'normal', 'high', 'very_high']
    )

    df['bmi_x_steps'] = df['bmi'].replace(0, np.nan) * df['step_count'].replace(0, np.nan)

    df['exercise_x_hr'] = (
        df['exercise_duration'].replace(0, np.nan) * df['heart_rate'].replace(0, np.nan)
    )

    df['sleep_x_stress'] = (
        df['sleep_duration'] *
        df['stress_level'].map({'low' : 1, 'medium' : 2,'high' : 3})
    )
    df['risk_score'] = (
        df['stress_level'].map({'low' : 1, 'medium' : 2, 'high' : 3}) *	
        df['sleep_quality'].map({'good' : 1, 'average' : 2, 'poor' : 3})	*
        df['physical_activity_level'].map({'active' : 1, 'moderate' : 2, 'sedentary' : 3}) *
        df['smoking_alcohol'].map({'no' : 1, 'occasional' : 2, 'yes' : 3})
    )
    return df

feature_engineer = FunctionTransformer(feature_engineering)

In [5]:
print(train.shape)
print(train.isna().sum())

(690088, 15)
id                             0
health_condition               0
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64


In [91]:
# Filling in Nans
from  sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import MissingIndicator
from sklearn.preprocessing import OneHotEncoder

numeric_cols = [
    'sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
    'step_count', 'exercise_duration', 'water_intake', 'calories_per_minute',
    'water_per_bmi', 'steps_per_minute', 'sleep_deprived', 'risk_score',
    'bmi_x_steps', 'exercise_x_hr',  'sleep_x_stress'
]
nominal_cols = [
    'diet_type', 'gender'
]
ordinal_cols = [
    'stress_level', 'sleep_quality', 'physical_activity_level',
    'smoking_alcohol', 'bmi_category', 'heart_rate_category'
]

transformer = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy = 'median', add_indicator = True)),
        ('scaler', StandardScaler())
                         ]), numeric_cols),
    ('nominal', Pipeline([
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown = 'ignore'))
    ]), nominal_cols),
    ('ordinal', Pipeline([
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('encoder', OrdinalEncoder(
            categories = [
                ['low', 'medium', 'high'],
                ['poor', 'average', 'good'],
                ['sedentary', 'moderate', 'active'],
                ['no', 'occasional', 'yes'],
                ['underweight', 'healthy_weight', 'over_weight', 'obese'],
                ['low', 'normal', 'high', 'very_high']
            ]
        ))
    ]), ordinal_cols),
    ('ordinal_missing_flags', MissingIndicator(features='missing-only'), ordinal_cols)
])

original_ordinal_cols = [
    'stress_level', 'sleep_quality', 'physical_activity_level',
    'smoking_alcohol'
]
train[original_ordinal_cols].apply(lambda x : x.unique())

,stress_level,sleep_quality,physical_activity_level,smoking_alcohol
0,high,average,sedentary,yes
1,low,poor,moderate,occasional
2,NaN,NaN,active,NaN
3,medium,good,NaN,no


In [7]:
# Dropping Columns 
train.drop(columns = {'id'}, inplace = True)


In [8]:
y_train = train['health_condition']
X_train = train.drop(columns = {'health_condition'})
print(y_train.value_counts(normalize = True))
X_train.head(6)

health_condition
at-risk      0.858675
unhealthy    0.083647
fit          0.057678
Name: proportion, dtype: float64


,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male
5,5.11,82.8,24.43,2290.0,2134.0,29.2,2.02,veg,low,poor,sedentary,occasional,male


In [57]:
# Modeling Imports
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier


# Scoring Metrics
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score


In [53]:
# Logistic Regression
lr = Pipeline([
    ('feature_engineering', feature_engineer),
    ('precrossing', transformer),
    ('lr', LogisticRegression(
        max_iter = 2000,
        class_weight = 'balanced'
    ))
])
lr_scores = cross_validate(
    estimator = lr,
    X = X_train,
    y = y_train, 
    cv = 5,
    scoring = {
        'accuracy': 'accuracy',
        'precision': make_scorer(precision_score, average = 'macro', zero_division = 0),
        'recall': make_scorer(recall_score, average = 'macro', zero_division = 0),
        'f1': make_scorer(f1_score, average = 'macro', zero_division = 0)
    },
    return_train_score = True
)
lr_summary = {
    'Accuracy' : lr_scores['test_accuracy'].mean(),
    'Precision' : lr_scores['test_precision'].mean(),
    'Recall' : lr_scores['test_recall'].mean(),
    'f1 value' : lr_scores['test_f1'].mean(),
    'Gap' : lr_scores['train_f1'].mean() - lr_scores['test_f1'].mean()
}

In [ ]:
lr_summary

In [54]:
# Random Forest
rf = Pipeline([
    ('feature_engineering', feature_engineer),
    ('preprocessor', transformer),
    ('rf', RandomForestClassifier(
        class_weight = 'balanced',
        random_state = 42,
    ))
])
rf_scores = cross_validate(
    estimator = rf,
    X = X_train,
    y = y_train,
    cv = 5,
    scoring = {
                'accuracy': 'accuracy',
        'precision': make_scorer(precision_score, average = 'macro', zero_division = 0),
        'recall': make_scorer(recall_score, average = 'macro', zero_division = 0),
        'f1': make_scorer(f1_score, average = 'macro', zero_division = 0)
    },
    return_train_score = True
)
rf_summary = {
    'Accuracy' : rf_scores['test_accuracy'].mean(),
    'Precision' : rf_scores['test_precision'].mean(),
    'Recall' : rf_scores['test_recall'].mean(),
    'f1 value' : rf_scores['test_f1'].mean(),
    'Gap' : rf_scores['train_f1'].mean() - rf_scores['test_f1'].mean()
}

In [55]:
# Linear Discriminant Analysis
lda = Pipeline([
    ('feature_engineering', feature_engineer),
    ('preprocessor', transformer),
    ('lda', LinearDiscriminantAnalysis(
        solver = 'lsqr'
    ))
])
lda_scores = cross_validate(
    estimator = lda,
    X = X_train,
    y = y_train,
    cv = 5,
    scoring = {
        'accuracy': 'accuracy',
        'precision': make_scorer(precision_score, average = 'macro', zero_division = 0),
        'recall': make_scorer(recall_score, average = 'macro', zero_division = 0),
        'f1': make_scorer(f1_score, average = 'macro', zero_division = 0)
    },
    return_train_score = True
)
lda_summary = {
    'Accuracy' : lda_scores['test_accuracy'].mean(),
    'Precision' : lda_scores['test_precision'].mean(),
    'Recall' : lda_scores['test_recall'].mean(),
    'f1 value' : lda_scores['test_f1'].mean(),
    'Gap' : lda_scores['train_f1'].mean() - lda_scores['test_f1'].mean()
}

In [ ]:
# KNN
knn = Pipeline([
    ('feature_engineering', feature_engineer),
    ('preprocessor', transformer),
    ('knn', KNeighborsClassifier(
        n_neighbors = 5,
        weights = 'distance'
    ))
])
knn_scores = cross_validate(
    estimator = knn,
    X = X_train,
    y = y_train,
    cv = 5,
    scoring = {
        'accuracy': 'accuracy',
        'precision': make_scorer(precision_score, average = 'macro', zero_division = 0),
        'recall': make_scorer(recall_score, average = 'macro', zero_division = 0),
        'f1': make_scorer(f1_score, average = 'macro', zero_division = 0)
    },
    return_train_score = True
)
knn_summary = {
    'Accuracy' : knn_scores['test_accuracy'].mean(),
    'Precision' : knn_scores['test_precision'].mean(),
    'Recall' : knn_scores['test_recall'].mean(),
    'f1 value' : knn_scores['test_f1'].mean(),
    'Gap' : knn_scores['train_f1'].mean() - knn_scores['test_f1'].mean()
}

In [12]:
# LGBM
lgbm = Pipeline([
    ('feature_engineering', feature_engineer),
    ('preprocessor', transformer),
    ('lgm', LGBMClassifier(
        objective = "multiclass",
        num_class = 3,
        learning_rate = 0.01,
        verbose = False,
        class_weight = 'balanced'
    ))
])
lgbm_scores = cross_validate(
    estimator = lgbm,
    X = X_train,
    y = y_train,
    cv = 5,
    scoring = {
        'accuracy': 'accuracy',
        'precision': make_scorer(precision_score, average = 'macro', zero_division = 0),
        'recall': make_scorer(recall_score, average = 'macro', zero_division = 0),
        'f1': make_scorer(f1_score, average = 'macro', zero_division = 0)
    },
    return_train_score = True
)
lgbm_summary = {
    'Accuracy' : lgbm_scores['test_accuracy'].mean(),
    'Precision' : lgbm_scores['test_precision'].mean(),
    'Recall' : lgbm_scores['test_recall'].mean(),
    'f1 value' : lgbm_scores['test_f1'].mean(),
    'Gap' : lgbm_scores['train_f1'].mean() - lgbm_scores['test_f1'].mean()
}

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/

In [67]:
print(y_train.unique())

['unhealthy' 'at-risk' 'fit']


In [70]:
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import StratifiedKFold
xgb = Pipeline([
    ('feature_engineering', feature_engineer),
    ('preprocessing', transformer),
    ('xgb', XGBClassifier(
        learning_rate = 0.001,
        random_state = 42,
        n_jobs = -1
    ))
])



results = {
    "train_accuracy": [],
    "train_precision": [],
    "train_recall": [],
    "train_f1": [],
    "test_accuracy": [],
    "test_precision": [],
    "test_recall": [],
    "test_f1": []
}

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
y_mapping = {
    'unhealthy' : 0,
    'at-risk' : 1,
    'fit' : 2
}
y_train_mapped = y_train.map(y_mapping)
xgb_scores = []
for train_idx, test_idx in cv.split(X_train, y_train_mapped):
    X_tr = X_train.iloc[train_idx]
    X_ts = X_train.iloc[test_idx]

    y_tr = y_train_mapped.iloc[train_idx]
    y_ts = y_train_mapped.iloc[test_idx]
    xgb_weights = compute_sample_weight(class_weight = 'balanced', y = y_tr)
    xgb.fit(X_tr, y_tr, xgb__sample_weight = xgb_weights)

    y_tr_pred = xgb.predict(X_tr)
    y_ts_pred = xgb.predict(X_ts)
    
    results["train_accuracy"].append(accuracy_score(y_tr, y_tr_pred))
    results["train_precision"].append(
        precision_score(y_tr, y_tr_pred, average="macro", zero_division=0)
    )
    results["train_recall"].append(
        recall_score(y_tr, y_tr_pred, average="macro", zero_division=0)
    )
    results["train_f1"].append(
        f1_score(y_tr, y_tr_pred, average="macro", zero_division=0)
    )

    # Test scores
    results["test_accuracy"].append(accuracy_score(y_ts, y_ts_pred))
    results["test_precision"].append(
        precision_score(y_ts, y_ts_pred, average="macro", zero_division=0)
    )
    results["test_recall"].append(
        recall_score(y_ts, y_ts_pred, average="macro", zero_division=0)
    )
    results["test_f1"].append(
        f1_score(y_ts, y_ts_pred, average="macro", zero_division=0)
    )

pd.DataFrame(results).agg(["mean", "std"]).T

,mean,std
train_accuracy,0.935726,0.001717
train_precision,0.797799,0.003515
train_recall,0.949040,0.000462
train_f1,0.859579,0.002642
test_accuracy,0.935568,0.001838
test_precision,0.797577,0.003976
test_recall,0.948296,0.001206
test_f1,0.859168,0.002759


In [56]:
ModelComparisons = pd.DataFrame(
    [
        lr_summary,
        rf_summary,
        lda_summary,
        lgbm_summary
    ],
    index = [
        'Logistic Regression',
        'Random Forest Classifier',
        'Linear Discriminant Analysis',
        'Light Gradient Boosting Machine'
    ]
)
ModelComparisons

,Accuracy,Precision,Recall,f1 value,Gap
Logistic Regression,0.849731,0.650192,0.915664,0.727055,0.000020
Random Forest Classifier,0.965371,0.960367,0.857393,0.903311,0.096621
Linear Discriminant Analysis,0.936739,0.933098,0.733776,0.808152,-0.000634
Light Gradient Boosting Machine,0.936847,0.801234,0.949136,0.861937,0.000141


In [ ]:
lr_summary

In [11]:
# Tuning LGBM
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

lgbm_tune = Pipeline([
    ('feature_engineering', feature_engineer),
    ('preprocessor', transformer),
    ('lgm', LGBMClassifier(
        objective="multiclass",
        num_class=3,
         verbose=-1,
        class_weight='balanced',
        random_state=42
    ))
])

param_grid = {
    'lgm__n_estimators': [100, 300, 500],
    'lgm__learning_rate': [0.01, 0.05, 0.1],
    'lgm__num_leaves': [15, 31, 63],
    'lgm__max_depth': [-1, 5, 10],
    'lgm__min_child_samples': [10, 20, 50],
    'lgm__subsample': [0.7, 0.9, 1.0],
    'lgm__colsample_bytree': [0.7, 0.9, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=lgbm_tune,
    param_distributions=param_grid,
    scoring='f1_macro', 
    cv = 5,
    n_iter=50,  
    n_jobs=-1,
    verbose= False
)

random_search.fit(X_train, y_train)

print("Best macro F1:", random_search.best_score_)
print("Best params:", random_search.best_params_)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/

Best macro F1: 0.8794583039406545
Best params: {'lgm__subsample': 0.7, 'lgm__num_leaves': 63, 'lgm__n_estimators': 500, 'lgm__min_child_samples': 20, 'lgm__max_depth': -1, 'lgm__learning_rate': 0.1, 'lgm__colsample_bytree': 1.0}


In [18]:
# Loading the Test Data
test = pd.read_csv('test.csv')
X_test = test.drop(columns = {'id'})

# Best params: {'lgm__subsample': 0.7, 'lgm__num_leaves': 63, 'lgm__n_estimators': 500, 'lgm__min_child_samples': 20, 'lgm__max_depth': -1, 'lgm__learning_rate': 0.1, 'lgm__colsample_bytree': 1.0}


In [19]:
# Test Data 

In [111]:
from sklearn.ensemble import VotingClassifier
lgbm = Pipeline([
    ('feature_engineering', feature_engineer),
    ('preprocessor', transformer),
    ('lgm', LGBMClassifier(
        objective = "multiclass",
        subsample = 0.5,
        num_leaves = 63,
        n_estimators = 500,
        min_child_samples = 20,
        max_depth = -1,
        num_class = 3,
        colsample_bytree = 1,
        learning_rate = 0.01,
        verbosity = -1,
        class_weight = 'balanced',
        n_jobs = -1
    ))
])
lgbm.fit(X_train, y_train)
predictions = lgbm.predict(X_test)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [103]:
submission = pd.DataFrame(
    data = {
        'id' : test['id'],
        'health_condition' : predictions
    }
)
submission['health_condition'] = submission['health_condition'].map({
    0 : 'unhealthy',
    1 : 'at-risk',
    2 : 'fit'
})
submission.head(6)

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
5,690093,fit


In [104]:
submission.to_csv('ten.csv', index = False)